# reduce-op-mean-divide — ex2: harmonic mean across ranks via 1/x transform + all_reduce + divide + invert

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reduce-op-mean-divide`. Running the final beacon cell reports progress against the `Distributed: reduce-op mean divide` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce-op mean divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-op-mean-divide`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-op-mean-divide"
DD_SUBTOPIC = "Distributed: reduce-op mean divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Harmonic mean via sum-then-divide-then-invert — quick refresher

`dist.ReduceOp` still has no `MEAN`, no `HARMONIC_MEAN`, no `GEOMETRIC_MEAN`. Every named mean is built from the same recipe: transform → `all_reduce(SUM)` → divide → inverse transform.

For the **harmonic mean** of N rank-local values `x_r > 0`:
```
H = N / (sum_r 1/x_r)
```
Distributed implementation:
```python
tensor = t.tensor([1.0 / local_value], dtype=t.float32)    # transform
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)              # sum 1/x_r
tensor /= world_size                                       # divide → mean of 1/x
harmonic = 1.0 / tensor.item()                             # inverse transform
```

**The in-place divide still matters.** Same `/=` vs `=` distinction as the arithmetic-mean case — keeps caller-held references stable.

**Why harmonic for rates / speeds.** Averaging samples/sec across ranks: arithmetic mean over-weights fast ranks (they finished more iterations). Harmonic mean weights by *time spent*, which is the throughput-correct aggregate.

### Exercise 2 — harmonic mean across ranks via 1/x transform + all_reduce + divide + invert

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the transform-sum-divide-invert recipe (1/x → `all_reduce(SUM)` → `/= world_size` → 1/.) to compute the harmonic mean of per-rank values on every rank.
> Keywords: harmonic-mean, all_reduce, transform-and-invert, throughput
> ```

**KCs targeted:** `transform-then-sum-then-divide-then-invert`, `throughput-aggregation`

Implement `ex2_harmonic_mean(rank, world_size, dist_module, local_value)`. Compute the harmonic mean `H = N / sum(1/x_r)` across ranks.

Steps inside the function:
1. Validate `local_value > 0` (harmonic mean is undefined on zero/negative inputs). Raise `ValueError` if not.
2. Build the reciprocal tensor: `tensor = t.tensor([1.0 / local_value], dtype=t.float32)`.
3. `dist_module.all_reduce(tensor, op=dist_module.ReduceOp.SUM)` — now `tensor[0]` is `sum_r 1/x_r`.
4. In-place divide: `tensor /= world_size` — now `tensor[0]` is the arithmetic mean of the reciprocals.
5. Invert: `harmonic = 1.0 / tensor.item()`.
6. Return `harmonic` — a Python float, identical on every rank.

**Use case.** Per-rank training throughput in samples/sec. Rank 0 might do 100 samples/sec, rank 1 might do 50; the ARITHMETIC mean (75) over-weights the fast rank. The HARMONIC mean = 2 / (1/100 + 1/50) = 66.67 — the time-weighted average, which is the throughput a downstream consumer actually sees.

Input: `rank`, `world_size` ints; `dist_module`; `local_value` positive float.
Output: float — harmonic mean, same on every rank.

In [ ]:
def ex2_harmonic_mean(rank: int, world_size: int, dist_module, local_value: float) -> float:
    """Harmonic mean across ranks via 1/x transform + all_reduce + divide + invert."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake


    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'


    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # rank-0-only side-effect log (per-call log lines for tests to inspect)
            self.side_effects = []

        def _reduce_op(self, bag, op):
            if op == 'SUM':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = out + x
                return out
            if op == 'MAX':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = _t_for_fake.maximum(out, x)
                return out
            if op == 'MIN':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = _t_for_fake.minimum(out, x)
                return out
            if op == 'PROD':
                out = bag[0].clone()
                for x in bag[1:]:
                    out = out * x
                return out
            raise ValueError(f'unknown fake op {op!r}')

        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            reduced = self._reduce_op(bag, op)
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()

        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                tensor.copy_(self._reduce_op(bag, op))
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()

        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()

        def gather(self, tensor, gather_list, dst):
            """Mock dist.gather — only dst's gather_list is populated."""
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('gth', [None] * self.world_size)
                self.scratch['gth'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['gth']
                for i, src_tensor in enumerate(bag):
                    gather_list[i].copy_(src_tensor)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('gth', None)
            self.barrier.wait()

        def all_gather(self, gather_list, tensor):
            """Mock dist.all_gather — every rank's gather_list is populated."""
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('agth', [None] * self.world_size)
                self.scratch['agth'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['agth']
            for i, src_tensor in enumerate(bag):
                gather_list[i].copy_(src_tensor)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('agth', None)
            self.barrier.wait()

        def barrier_op(self):
            self.barrier.wait()


    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size

        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.gather = lambda tensor, gather_list, dst: world.gather(tensor, gather_list, dst)
            fake_dist.all_gather = lambda gather_list, tensor: world.all_gather(gather_list, tensor)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.init_process_group = lambda **kw: None
            fake_dist.destroy_process_group = lambda: None
            try:
                worker_fn(rank, world_size, fake_dist, world, *extra_args)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())

        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world.results


    # Classic case: throughput rank 0=100, rank 1=50.
    # H = 2 / (1/100 + 1/50) = 2 / 0.03 = 66.6667.
    _vals2 = [100.0, 50.0]

    def _worker2(rank, world_size, dist_module, world):
        world.results[rank] = ex2_harmonic_mean(rank, world_size, dist_module, _vals2[rank])

    results = _run_fake_world(_worker2, 2)
    expected = 2.0 / (1.0/100.0 + 1.0/50.0)   # = 200/3 ≈ 66.667
    for rank, r in enumerate(results):
        assert r is not None
        assert abs(r - expected) < 1e-3, (
            f'rank {rank}: got {r}, expected {expected}. '
            f'If you got 75.0, you computed the ARITHMETIC mean — '
            f'remember to transform/invert AROUND the reduce.'
        )

    # Identical values — harmonic mean degenerates to that value.
    def _worker_same(rank, world_size, dist_module, world):
        world.results[rank] = ex2_harmonic_mean(rank, world_size, dist_module, 12.0)

    results_same = _run_fake_world(_worker_same, 4)
    for rank, r in enumerate(results_same):
        assert abs(r - 12.0) < 1e-4, f'identical-values rank {rank}: got {r}'

    # Three ranks, asymmetric.
    _vals3 = [1.0, 2.0, 4.0]

    def _worker3(rank, world_size, dist_module, world):
        world.results[rank] = ex2_harmonic_mean(rank, world_size, dist_module, _vals3[rank])

    results3 = _run_fake_world(_worker3, 3)
    expected3 = 3.0 / (1.0/1.0 + 1.0/2.0 + 1.0/4.0)   # = 3 / 1.75 = 12/7 ≈ 1.714
    for rank, r in enumerate(results3):
        assert abs(r - expected3) < 1e-4, f'3-rank rank {rank}: got {r}, expected {expected3}'

    # Single-rank degenerate — harmonic mean of one value is itself.
    def _worker1(rank, world_size, dist_module, world):
        world.results[rank] = ex2_harmonic_mean(rank, world_size, dist_module, 9.0)

    results1 = _run_fake_world(_worker1, 1)
    assert abs(results1[0] - 9.0) < 1e-5

    # Zero input must raise.
    def _worker_zero(rank, world_size, dist_module, world):
        try:
            ex2_harmonic_mean(rank, world_size, dist_module, 0.0)
            world.results[rank] = 'no-raise'
        except ValueError:
            world.results[rank] = 'raised'
        except Exception as e:
            world.results[rank] = f'wrong-exc:{type(e).__name__}'

    # Single-rank world so the lack of a paired call doesn't matter.
    rz = _run_fake_world(_worker_zero, 1)
    assert rz[0] == 'raised', f'zero input must raise ValueError, got {rz[0]!r}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_harmonic_mean(rank: int, world_size: int, dist_module, local_value: float) -> float:
    if local_value <= 0:
        raise ValueError(f'harmonic mean undefined for non-positive input: {local_value}')
    tensor = t.tensor([1.0 / local_value], dtype=t.float32)
    dist_module.all_reduce(tensor, op=dist_module.ReduceOp.SUM)
    tensor /= world_size
    return 1.0 / tensor.item()
```

**The 'transform-aggregate-invert' shape is general.** Geometric mean = log → SUM → /= N → exp. Quadratic mean = square → SUM → /= N → sqrt. All built from `all_reduce(SUM)` + in-place divide + element-wise nonlinearities.

**Why validate `> 0`.** `1.0 / 0.0` is `inf` in IEEE 754; `1.0 / -2.0` flips sign and silently produces a finite-but-wrong answer. Both are footguns the test explicitly checks against.

**ReduceOp.SUM works on the transformed values, not the originals.** This is the key insight ex1 hints at and ex2 makes concrete: the missing `ReduceOp.MEAN` is just sugar for SUM + divide. Any named mean is SUM + divide of the right transformed values.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()